In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# Locate the original data files.
data_dir = Path(
    "/Users/daviddejonghe/Documents/Psychologie/"
    "Masterproef/simulationsHmfc/Data"
)

task_data = pd.read_csv(data_dir / "PerceptualTaskData.csv")
symptom_data = pd.read_csv(data_dir / "SymptomData.csv")

display(task_data.head())
display(symptom_data.head())



,IDcode,Age,Sex,Trial,Response,Stimulus,Confidence
0,4676165,67,1,1,0,16,1
1,4676165,67,1,2,0,8,2
2,4676165,67,1,3,1,56,3
3,4676165,67,1,4,1,32,3
4,4676165,67,1,5,0,48,3


,IDcode,Age,Sex,AnxiousDepression,Compulsivity,SocialWithdrawal
0,4676165,67,1,0.997218,-1.174643,0.503585
1,4676332,28,1,-0.488935,-1.178822,0.197481
2,4676373,19,1,-1.180051,-1.219396,-1.101509
3,4676552,24,1,-0.280664,-0.475714,-0.558212
4,4676560,34,1,2.167959,1.647215,0.403726


In [11]:
# Check that participant IDs are present and unique in the symptom table.
assert task_data["IDcode"].notna().all()
assert symptom_data["IDcode"].notna().all()
assert symptom_data["IDcode"].is_unique

# Check that both files contain the same participants.
assert set(task_data["IDcode"]) == set(symptom_data["IDcode"])

# Check that each participant-trial combination occurs only once.
assert not task_data.duplicated(["IDcode", "Trial"]).any()

# Check the task variables needed for subsequent modelling.
assert task_data[["Trial", "Response", "Stimulus"]].notna().all().all()
assert task_data["Response"].isin([0, 1]).all()

print(f"Participants: {task_data['IDcode'].nunique()}")
print(f"Trials: {len(task_data)}")

Participants: 1098
Trials: 155088


In [12]:
# Choose the symptom scale and the target number of participants per group.
scale = "SocialWithdrawal"
group_size = 10
random_seed = 2026

assert group_size > 0
assert symptom_data[scale].notna().all()
assert np.isfinite(symptom_data[scale]).all()

# Sort participants from lowest to highest symptom score.
# Participant ID provides a reproducible ordering when scores are tied.
ranked_participants = (
    symptom_data
    .sort_values([scale, "IDcode"])
    .reset_index(drop=True)
)

display(ranked_participants[["IDcode", scale]].head(20))

,IDcode,SocialWithdrawal
0,1235242,-2.099924
1,2391757,-2.071829
2,1096800,-2.052834
3,1235485,-2.029991
4,1030377,-1.999375
5,1235377,-1.992363
6,5479352,-1.971656
7,4676782,-1.948856
8,5701810,-1.920629
9,1069986,-1.911915


In [13]:
# Assign consecutive sets of 10 participants to the same group.
# Keep the final group even if it contains fewer than 10 participants.
ranked_participants["superparticipant"] = (
    np.arange(len(ranked_participants)) // group_size
) + 1

display(
    ranked_participants[
        ["IDcode", scale, "superparticipant"]
    ].head(25)
)

# Inspect the number of participants in each group.
group_sizes = ranked_participants.groupby("superparticipant").size()

print(f"Superparticipants: {len(group_sizes)}")
print(group_sizes.value_counts().sort_index())

,IDcode,SocialWithdrawal,superparticipant
0,1235242,-2.099924,1
1,2391757,-2.071829,1
2,1096800,-2.052834,1
3,1235485,-2.029991,1
4,1030377,-1.999375,1
5,1235377,-1.992363,1
6,5479352,-1.971656,1
7,4676782,-1.948856,1
8,5701810,-1.920629,1
9,1069986,-1.911915,1


Superparticipants: 110
8       1
10    109
Name: count, dtype: int64


In [14]:
# Give each original participant equal weight in the group symptom score.
group_scores = (
    ranked_participants
    .groupby("superparticipant", as_index=False)
    .agg(
        symptom_mean=(scale, "mean"),
        symptom_min=(scale, "min"),
        symptom_max=(scale, "max"),
        n_participants=("IDcode", "size"),
    )
)

# Record which scale was used to construct these groups.
group_scores["scale"] = scale

display(group_scores.head())

,superparticipant,symptom_mean,symptom_min,symptom_max,n_participants,scale
0,1,-1.999937,-2.099924,-1.911915,10,SocialWithdrawal
1,2,-1.821761,-1.879921,-1.759961,10,SocialWithdrawal
2,3,-1.684115,-1.748834,-1.628189,10,SocialWithdrawal
3,4,-1.591373,-1.602802,-1.570339,10,SocialWithdrawal
4,5,-1.546098,-1.568960,-1.524014,10,SocialWithdrawal


In [15]:
# Create a participant-to-group mapping.
membership = ranked_participants[
    ["IDcode", "superparticipant"]
].copy()

# Randomize participant order within each group reproducibly.
rng = np.random.default_rng(random_seed)
membership["random_order"] = rng.random(len(membership))

membership = (
    membership
    .sort_values(["superparticipant", "random_order"])
    .reset_index(drop=True)
)

membership["participant_order"] = (
    membership.groupby("superparticipant").cumcount() + 1
)

membership = membership.drop(columns="random_order")

display(membership.head(15))

,IDcode,superparticipant,participant_order
0,4676782,1,1
1,1235242,1,2
2,1069986,1,3
3,1030377,1,4
4,1235485,1,5
5,1096800,1,6
6,2391757,1,7
7,5701810,1,8
8,1235377,1,9
9,5479352,1,10


In [16]:
# Attach each participant's group and within-group position to every trial.
pooled_data = task_data.merge(
    membership,
    on="IDcode",
    how="left",
    validate="many_to_one",
)

# Preserve the original trial order within each participant.
pooled_data = (
    pooled_data
    .sort_values(["superparticipant", "participant_order", "Trial"])
    .reset_index(drop=True)
)

# Number trials consecutively within each superparticipant.
pooled_data["pooled_trial"] = (
    pooled_data.groupby("superparticipant").cumcount() + 1
)

# Mark the first trial of each original participant.
# This is metadata only: it does not reset the latent process in hMFC.
pooled_data["segment_start"] = (
    pooled_data.groupby("IDcode").cumcount() == 0
)

display(
    pooled_data[
        [
            "superparticipant",
            "IDcode",
            "Trial",
            "pooled_trial",
            "Response",
            "Stimulus",
            "segment_start",
        ]
    ].head(20)
)

,superparticipant,IDcode,Trial,pooled_trial,Response,Stimulus,segment_start
0,1,4676782,1,1,0,16,True
1,1,4676782,2,2,1,-64,False
2,1,4676782,3,3,0,-56,False
3,1,4676782,4,4,0,0,False
4,1,4676782,5,5,0,40,False
5,1,4676782,6,6,1,-32,False
6,1,4676782,7,7,1,72,False
7,1,4676782,8,8,1,8,False
8,1,4676782,9,9,1,64,False
9,1,4676782,10,10,0,-64,False


In [17]:
# Check that pooling preserved every trial and participant.
assert len(pooled_data) == len(task_data)
assert pooled_data["IDcode"].nunique() == task_data["IDcode"].nunique()
assert pooled_data["superparticipant"].notna().all()
assert not pooled_data.duplicated(["IDcode", "Trial"]).any()

# Check that each participant belongs to exactly one superparticipant.
assert (
    pooled_data.groupby("IDcode")["superparticipant"]
    .nunique()
    .eq(1)
    .all()
)

# Verify that all original columns and values are unchanged.
sort_columns = ["IDcode", "Trial"]

pd.testing.assert_frame_equal(
    task_data.sort_values(sort_columns).reset_index(drop=True),
    pooled_data[task_data.columns]
    .sort_values(sort_columns)
    .reset_index(drop=True),
)

# Add the total number of trials to the group summary.
trial_counts = (
    pooled_data.groupby("superparticipant")
    .size()
    .rename("n_trials")
    .reset_index()
)

group_summary = group_scores.merge(
    trial_counts,
    on="superparticipant",
    validate="one_to_one",
)

display(group_summary.head())

# Inspect the participant boundaries in the first superparticipant.
display(
    pooled_data.loc[
        (pooled_data["superparticipant"] == 1)
        & pooled_data["segment_start"],
        ["IDcode", "Trial", "pooled_trial"],
    ]
)

,superparticipant,symptom_mean,symptom_min,symptom_max,n_participants,scale,n_trials
0,1,-1.999937,-2.099924,-1.911915,10,SocialWithdrawal,1408
1,2,-1.821761,-1.879921,-1.759961,10,SocialWithdrawal,1440
2,3,-1.684115,-1.748834,-1.628189,10,SocialWithdrawal,1408
3,4,-1.591373,-1.602802,-1.570339,10,SocialWithdrawal,1440
4,5,-1.546098,-1.568960,-1.524014,10,SocialWithdrawal,1408


,IDcode,Trial,pooled_trial
0,4676782,1,1
152,1235242,1,153
288,1069986,1,289
424,1030377,1,425
560,1235485,1,561
696,1096800,1,697
832,2391757,1,833
968,5701810,1,969
1120,1235377,1,1121
1256,5479352,1,1257


In [18]:
# Create a separate output directory for this scale and group size.
output_dir = Path.cwd() / f"pooling_{scale}_n{group_size}"
output_dir.mkdir(parents=True, exist_ok=True)

# Running this cell again overwrites files with the same names.
pooled_data.to_csv(output_dir / "pooled_task_data.csv", index=False)
group_summary.to_csv(output_dir / "group_scores.csv", index=False)
membership.to_csv(output_dir / "membership.csv", index=False)

print(f"Files saved to: {output_dir.resolve()}")

Files saved to: /Users/daviddejonghe/Documents/Psychologie/Masterproef/simulationsHmfc/super_participants/pooling_SocialWithdrawal_n10
